!pip install numpy==1.23.5 scikit-learn==1.2.2

In [ ]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

In [ ]:
import joblib

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

print("All files loaded successfully ")

In [ ]:
print("Pipeline Stages:")
stages = ["Ingest", "Anonymise", "Embed", "FAISS", "Groq Insights", "Export"]

for i, stage in enumerate(stages, 1):
    print(f"Step {i}: {stage}")

In [ ]:
import pandas as pd

df = pd.read_excel("balanced_edufeed_dataset.xlsx")

print("Total rows:", len(df))
df.head()

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.columns)

In [ ]:
import re
import pandas as pd

def contains_sensitive(text):
    if pd.isna(text):
        return False

    email = re.search(r'\S+@\S+', str(text))
    numbers = re.search(r'\d{10}', str(text))

    return bool(email or numbers)

In [ ]:
df["sensitive_flag"] = df["comments"].apply(contains_sensitive)

In [ ]:
col = df.columns[0]  # or manually set correct name

df["sensitive_flag"] = df[col].apply(contains_sensitive)

print("Sensitive data found:", df["sensitive_flag"].sum())

In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu

In [ ]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

In [ ]:
from sentence_transformers import SentenceTransformer
model_embed = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

In [ ]:
texts = df["comments"].astype(str).tolist()

In [ ]:
import time

start = time.time()

embeddings = model_embed.encode(texts[:500])  # use sample first

end = time.time()

print("Embedding time:", end - start)

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
import numpy as np

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors")

In [ ]:
query = embeddings[0].reshape(1, -1)

start = time.time()
D, I = index.search(query, k=5)
end = time.time()

print("Search time:", end - start)

In [ ]:
import json

results = {
    "sample_results": I.tolist(),
    "distances": D.tolist()
}

with open("results.json", "w") as f:
    json.dump(results, f)

print("Results exported")

In [ ]:
!ls

In [ ]:
import json

with open("results.json", "r") as f:
    data = json.load(f)

print("Export verified ")

In [ ]:
df.head()
vectorizer
model

In [ ]:
feature_names = vectorizer.get_feature_names_out()
print(f"Total features:",{len(feature_names)})

In [ ]:
important_words = ["hard", "helpful", "easy", "difficult", "pass"]

for word in important_words:
    if word in feature_names:
        print(f"{word} is in vocabulary")
    else:
        print(f"{word} NOT found")

In [ ]:
df["comments"].isna().sum()

In [ ]:
df["comments"] = df["comments"].fillna("")

In [ ]:
df["comments"] = df["comments"].astype(str)

In [ ]:
feature_names = vectorizer.get_feature_names_out()

X = vectorizer.transform(df["comments"])

import numpy as np
scores = np.asarray(X.mean(axis=0)).flatten()

top_indices = scores.argsort()[-10:][::-1]

print("Top important words:\n")
for i in top_indices:
    print(feature_names[i])

In [ ]:
# Ensure text is clean (IMPORTANT)
df["comments"] = df["comments"].fillna("").astype(str)

# Transform text to features
X = vectorizer.transform(df["comments"])

# True labels
y_true = df["sentiment_label"]

# Predictions from Logistic Regression model
y_pred = model.predict(X)

In [ ]:
from sklearn.metrics import classification_report

print("Classification Report:\n")
print(classification_report(y_true, y_pred, digits=2))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy:{accuracy:.2f}")

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

In [ ]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

In [ ]:
classifier = pipeline("zero-shot-classification", model="microsoft/deberta-v3-base")

In [ ]:
labels = ["positive", "neutral", "negative"]

In [ ]:
test_sentences = [
    "The course is not bad at all",
    "Assignments are killing me but I learned a lot",
    "Yeah great teaching... not really",
    "The faculty is okay, nothing special"
]

for text in test_sentences:
    result = classifier(text, candidate_labels=labels)
    print("\nText:", text)
    print("Prediction:", result["labels"][0])

In [ ]:
results = []

sample_texts = df["comments"].sample(5)

for text in sample_texts:
    lr_pred = model.predict(vectorizer.transform([text]))[0]
    bert_pred = classifier(text, candidate_labels=labels)["labels"][0]

    results.append((text, lr_pred, bert_pred))

In [ ]:
for text, lr, bert in results:
    print("\nText:", text[:100])
    print("Logistic Regression:", lr)
    print("DeBERTa:", bert)

In [ ]:
mismatch = [r for r in results if r[1] != r[2]]

print("\nTotal mismatches:", len(mismatch))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))